# Notebook 7: OOD/OE Quality Human Review

Bu notebook hazir runtime dataset icindeki `ood/` ve `oe/` havuzlarini human-in-loop sekilde audit eder.

Kontroller:
- exact SHA-256 overlap
- perceptual near-duplicate adaylari
- slice/folder semantik supheleri

Notebook otomatik silmez. Reviewer `review_decisions.csv` icinde sadece emin oldugu satirlara `decision=quarantine` yazar; apply hucresi bu dosyalari quarantine klasorune tasir.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
CLONE_TARGET = Path(os.environ.get('AADS_REPO_CLONE_TARGET', '/content/bitirmeprojesi'))

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'scripts').is_dir() and (path / 'config').is_dir()

def _resolve_repo_root() -> Path:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if raw:
            candidate = Path(raw).expanduser().resolve()
            if _is_repo_root(candidate):
                return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents, CLONE_TARGET, Path('/content/bitirmeprojesi')]:
        candidate = candidate.expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if _is_repo_root(CLONE_TARGET):
        return CLONE_TARGET.resolve()
    raise FileNotFoundError('Repo root bulunamadi. AADS_REPO_ROOT set edin veya clone hedefini kontrol edin.')

REPO_ROOT = _resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'[SETUP] repo={REPO_ROOT}')


In [ ]:
# Parametreler
RUN_ALL_DATASETS = True
ADAPTER_KEY = 'tomato__leaf'

# Batch modunda PREPARED_ROOT altindaki ood/ veya oe/ iceren tum datasetler auditlenir.
PREPARED_ROOT = REPO_ROOT / 'data' / 'prepared_runtime_datasets'

# Tek dataset modunda bos birakilirsa data/prepared_runtime_datasets/<ADAPTER_KEY> kullanilir.
DATASET_ROOT = ''

# Audit artifactleri buraya yazilir.
OUTPUT_DIR = ''

# 6: konservatif near-duplicate adayi. Daha yuksek deger daha cok review satiri uretir.
NEAR_DUPLICATE_HAMMING = 6

dataset_root = Path(DATASET_ROOT or (PREPARED_ROOT / ADAPTER_KEY)).expanduser()
output_dir = Path(OUTPUT_DIR or (REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_audit')).expanduser()
if not RUN_ALL_DATASETS:
    output_dir = output_dir / ADAPTER_KEY
print(f'[PARAM] run_all={RUN_ALL_DATASETS}')
print(f'[PARAM] prepared_root={PREPARED_ROOT}')
print(f'[PARAM] dataset_root={dataset_root}')
print(f'[PARAM] output_dir={output_dir}')


In [ ]:
import json

cmd = [sys.executable, 'scripts/audit_ood_oe_quality.py']
if RUN_ALL_DATASETS:
    cmd += ['--all', '--prepared-root', str(PREPARED_ROOT)]
else:
    cmd += ['--dataset-root', str(dataset_root), '--dataset-key', ADAPTER_KEY]
cmd += [
    '--output-dir', str(output_dir),
    '--near-duplicate-hamming', str(NEAR_DUPLICATE_HAMMING),
]
completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(f'Audit failed with exit code {completed.returncode}')

summary_name = 'batch_summary.json' if RUN_ALL_DATASETS else 'summary.json'
summary = json.loads((output_dir / summary_name).read_text(encoding='utf-8'))
if RUN_ALL_DATASETS:
    print('[AUDIT] batch summary CSV:', output_dir / 'batch_summary.csv')
    print('[AUDIT] per-dataset review CSVs:', output_dir / '<dataset_key>' / 'review_decisions.csv')
else:
    print('[AUDIT] review CSV:', output_dir / 'review_decisions.csv')
    print('[AUDIT] markdown report:', output_dir / 'review_report.md')

try:
    import pandas as pd
    from IPython.display import display
    preview_csv = output_dir / 'batch_summary.csv' if RUN_ALL_DATASETS else output_dir / 'review_decisions.csv'
    issues_df = pd.read_csv(preview_csv)
    display(issues_df.head(50))
except Exception as exc:
    print('[AUDIT] Pandas preview unavailable:', exc)


## Human Review

1. `review_decisions.csv` dosyasini acin.
2. Sadece emin oldugunuz satirlarda `decision` kolonunu `quarantine` yapin.
3. `target_path` hangi dosyanin tasinacagini belirler. Emin degilseniz bos birakin veya `ignore` yazin.
4. Apply hucresi dosya silmez; hedefleri `QUARANTINE_ROOT` altina tasir.


In [ ]:
APPLY_REVIEW_DECISIONS = False
# Batch modunda once REVIEW_ADAPTER_KEY secilir; her adapterin CSV'i ayridir.
REVIEW_ADAPTER_KEY = ADAPTER_KEY
active_dataset_root = PREPARED_ROOT / REVIEW_ADAPTER_KEY if RUN_ALL_DATASETS else dataset_root
DECISIONS_CSV = (output_dir / REVIEW_ADAPTER_KEY / 'review_decisions.csv') if RUN_ALL_DATASETS else (output_dir / 'review_decisions.csv')
QUARANTINE_ROOT = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_quarantine' / REVIEW_ADAPTER_KEY

if not APPLY_REVIEW_DECISIONS:
    print('[APPLY] Kapali. Batch modunda REVIEW_ADAPTER_KEY secip ilgili review_decisions.csv duzenlendikten sonra APPLY_REVIEW_DECISIONS=True yapin.')
else:
    cmd = [
        sys.executable,
        'scripts/audit_ood_oe_quality.py',
        '--dataset-root', str(active_dataset_root),
        '--apply-decisions', str(DECISIONS_CSV),
        '--quarantine-root', str(QUARANTINE_ROOT),
    ]
    completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Apply failed with exit code {completed.returncode}')


In [ ]:
# Apply sonrasi yeniden audit etmek icin bu hucresi calistirin.
RERUN_AFTER_APPLY = False
if RERUN_AFTER_APPLY:
    rerun_output = output_dir / 'post_apply'
    cmd = [sys.executable, 'scripts/audit_ood_oe_quality.py']
    if RUN_ALL_DATASETS:
        cmd += ['--all', '--prepared-root', str(PREPARED_ROOT)]
    else:
        cmd += ['--dataset-root', str(dataset_root), '--dataset-key', ADAPTER_KEY]
    cmd += [
        '--output-dir', str(rerun_output),
        '--near-duplicate-hamming', str(NEAR_DUPLICATE_HAMMING),
    ]
    completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Rerun failed with exit code {completed.returncode}')
    print('[RERUN] output:', rerun_output)
